# BFSI Agent Router
Reads classified complaints from `complaints.duckdb`, routes each to correct team, writes decisions to `routing.duckdb`.

**Run order:** Cell 1 → 2 → 3 → 4

## Cell 1 — Config

In [ ]:
VLLM_HOST     = "http://localhost:8000"      # update to AMD VM IP
VLLM_BASE_URL = f"{VLLM_HOST}/v1"
VLLM_MODEL    = "Qwen2.5-7B-Instruct"         # match --served-model-name
VLLM_API_KEY  = "abc-123"

COMPLAINTS_DB = "complaints.duckdb"            # source — written by classifier notebook
ROUTING_DB    = "routing.duckdb"               # destination — written by this notebook

TEAM_PROFILES = {
    "Fraud Response Team": {
        "handles":    ["unauthorized transactions", "card fraud", "account takeover", "phishing"],
        "escalation": "Head of Fraud",
        "sla_hours":  24,
        "email":      "fraud-alerts@bank.in",
    },
    "Card Operations": {
        "handles":    ["card blocked", "card declined", "duplicate charge", "card replacement"],
        "escalation": "Card Ops Manager",
        "sla_hours":  48,
        "email":      "card-ops@bank.in",
    },
    "KYC Team": {
        "handles":    ["KYC pending", "document verification", "account opening delay", "video KYC"],
        "escalation": "KYC Head",
        "sla_hours":  48,
        "email":      "kyc@bank.in",
    },
    "Loan Servicing": {
        "handles":    ["EMI discrepancy", "loan prepayment", "interest rate", "home loan"],
        "escalation": "Loans Manager",
        "sla_hours":  120,
        "email":      "loans@bank.in",
    },
    "Digital Banking": {
        "handles":    ["UPI failure", "net banking", "mobile app", "NEFT/IMPS"],
        "escalation": "Digital Head",
        "sla_hours":  120,
        "email":      "digital@bank.in",
    },
    "Branch Support": {
        "handles":    ["ATM cash discrepancy", "branch service", "fixed deposit", "passbook"],
        "escalation": "Branch Manager",
        "sla_hours":  120,
        "email":      "branch@bank.in",
    },
    "Insurance Claims": {
        "handles":    ["life insurance claim", "health insurance", "cashless rejection", "policy lapse"],
        "escalation": "Claims Head",
        "sla_hours":  48,
        "email":      "insurance@bank.in",
    },
    "General Support": {
        "handles":    ["statement", "mobile number update", "feedback", "general inquiry"],
        "escalation": "Support Manager",
        "sla_hours":  240,
        "email":      "support@bank.in",
    },
}

ROUTER_SYSTEM = """You are a BFSI complaint routing agent. Reason step-by-step and decide
the best team to handle this complaint.

Teams and what they handle:
{team_profiles}

Return ONLY a valid JSON object with these fields:
- routed_team: exact team name from the list above
- escalate: true or false
- escalate_to: escalation contact name if escalate=true, else null
- action: specific instruction for the team (1-2 sentences)
- routing_reason: why this team was chosen (1-2 sentences)
- confidence: high | medium | low

No markdown. No text outside JSON."""

print("Config loaded.")

## Cell 2 — Routing DB setup

In [ ]:
import duckdb

def get_routing_conn():
    return duckdb.connect(ROUTING_DB)

def init_routing_db():
    conn = get_routing_conn()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS routing_decisions (
            routing_id           VARCHAR PRIMARY KEY,
            complaint_id         VARCHAR NOT NULL,
            customer_id          VARCHAR,
            cust_name            VARCHAR,
            complaint            TEXT,
            category             VARCHAR,
            priority             VARCHAR,
            sentiment            VARCHAR,
            original_team        VARCHAR,
            routed_team          VARCHAR,
            escalate             BOOLEAN,
            escalate_to          VARCHAR,
            action               TEXT,
            routing_reason       TEXT,
            confidence           VARCHAR,
            team_email           VARCHAR,
            sla_hours            INTEGER,
            complaint_created_at TIMESTAMP,
            routing_status       VARCHAR DEFAULT 'routed',
            routed_at            TIMESTAMP DEFAULT current_timestamp,
            latency_ms           INTEGER,
            model_used           VARCHAR
        )
    """)
    conn.close()
    print("Routing DB initialised:", ROUTING_DB)

def insert_routing_decision(d):
    conn = get_routing_conn()
    conn.execute("""
        INSERT OR REPLACE INTO routing_decisions (
            routing_id, complaint_id, customer_id, cust_name, complaint,
            category, priority, sentiment, original_team, routed_team,
            escalate, escalate_to, action, routing_reason, confidence,
            team_email, sla_hours, complaint_created_at,
            routing_status, routed_at, latency_ms, model_used
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                  'routed', current_timestamp, ?, ?)
    """, [
        d["routing_id"],    d["complaint_id"],
        d["customer_id"],   d["cust_name"],
        d["complaint"],     d["category"],
        d["priority"],      d["sentiment"],
        d["original_team"], d["routed_team"],
        d["escalate"],      d["escalate_to"],
        d["action"],        d["routing_reason"],
        d["confidence"],    d["team_email"],
        d["sla_hours"],     d["complaint_created_at"],
        d["latency_ms"],    d["model_used"],
    ])
    conn.close()

def fetch_unrouted():
    """Fetch complaints not yet in routing_decisions."""
    src = duckdb.connect(COMPLAINTS_DB, read_only=True)
    rows = src.execute("""
        SELECT complaint_id, customer_id, cust_name, complaint,
               category, priority, sentiment, team, created_at
        FROM complaints
        ORDER BY priority ASC, created_at ASC
    """).fetchall()
    src.close()

    cols = ["complaint_id", "customer_id", "cust_name", "complaint",
            "category", "priority", "sentiment", "team", "created_at"]
    all_complaints = [dict(zip(cols, r)) for r in rows]

    # exclude already routed
    try:
        rtr = duckdb.connect(ROUTING_DB, read_only=True)
        already = {r[0] for r in rtr.execute(
            "SELECT complaint_id FROM routing_decisions").fetchall()}
        rtr.close()
    except Exception:
        already = set()

    pending = [c for c in all_complaints if c["complaint_id"] not in already]
    print(f"Total in complaints DB : {len(all_complaints)}")
    print(f"Already routed         : {len(already)}")
    print(f"Pending routing        : {len(pending)}")
    return pending

init_routing_db()
print("DB functions ready.")

## Cell 3 — Router agent functions

In [ ]:
import time, uuid, json
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

def route_one(c):
    """Route a single complaint dict. Returns routing decision dict."""
    routing_id   = "RTR-" + str(uuid.uuid4())[:8].upper()
    profiles_str = json.dumps(
        {k: v["handles"] for k, v in TEAM_PROFILES.items()}, indent=2
    )
    system_prompt = ROUTER_SYSTEM.format(team_profiles=profiles_str)
    user_msg = (
        f"Complaint ID  : {c['complaint_id']}\n"
        f"Customer      : {c['cust_name']}\n"
        f"Complaint     : {c['complaint']}\n"
        f"Category      : {c['category']}\n"
        f"Priority      : {c['priority']}\n"
        f"Sentiment     : {c['sentiment']}\n"
        f"Created at    : {c['created_at']}\n"
        f"Initial team  : {c['team']}"
    )
    t0       = time.time()
    response = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.1,
        max_tokens=400,
    )
    latency_ms = int((time.time() - t0) * 1000)
    raw        = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw.replace("```json", "").replace("```", "").strip())
    except json.JSONDecodeError:
        raise ValueError(f"Non-JSON response: {raw[:200]}")

    routed_team  = result.get("routed_team", c["team"])
    team_profile = TEAM_PROFILES.get(routed_team, {})

    return {
        "routing_id":           routing_id,
        "complaint_id":         c["complaint_id"],
        "customer_id":          c["customer_id"],
        "cust_name":            c["cust_name"],
        "complaint":            c["complaint"],
        "category":             c["category"],
        "priority":             c["priority"],
        "sentiment":            c["sentiment"],
        "original_team":        c["team"],
        "routed_team":          routed_team,
        "escalate":             result.get("escalate", False),
        "escalate_to":          result.get("escalate_to"),
        "action":               result.get("action", ""),
        "routing_reason":       result.get("routing_reason", ""),
        "confidence":           result.get("confidence", "medium"),
        "team_email":           team_profile.get("email", ""),
        "sla_hours":            team_profile.get("sla_hours", 120),
        "complaint_created_at": c["created_at"],
        "latency_ms":           latency_ms,
        "model_used":           VLLM_MODEL,
    }

def run_router(rerun=False):
    """
    Route complaints from COMPLAINTS_DB into ROUTING_DB.
    rerun=True  — re-route all complaints including already routed ones.
    rerun=False — only route complaints not yet in routing_decisions.
    """
    if rerun:
        src  = duckdb.connect(COMPLAINTS_DB, read_only=True)
        rows = src.execute("""
            SELECT complaint_id, customer_id, cust_name, complaint,
                   category, priority, sentiment, team, created_at
            FROM complaints ORDER BY priority ASC, created_at ASC
        """).fetchall()
        src.close()
        cols       = ["complaint_id", "customer_id", "cust_name", "complaint",
                      "category", "priority", "sentiment", "team", "created_at"]
        complaints = [dict(zip(cols, r)) for r in rows]
    else:
        complaints = fetch_unrouted()

    if not complaints:
        print("Nothing to route. Run fetch_unrouted() to check status.")
        return []

    print(f"\nRouting {len(complaints)} complaint(s) via {VLLM_MODEL}...\n")
    decisions, failed = [], 0

    for i, c in enumerate(complaints):
        try:
            d = route_one(c)
            insert_routing_decision(d)
            decisions.append(d)
            flag = " ⚑ ESCALATE" if d["escalate"] else ""
            print(
                f"[{i+1}/{len(complaints)}] {c['complaint_id']} "
                f"| {c['created_at']} "
                f"→ {d['routed_team']} [{d['confidence']}]{flag} "
                f"({d['latency_ms']}ms)"
            )
        except Exception as e:
            print(f"[{i+1}/{len(complaints)}] FAILED {c['complaint_id']}: {e}")
            failed += 1

    print(f"\nDone. Routed: {len(decisions)} | Failed: {failed}")
    return decisions

print("Router agent ready.")

## Cell 4 — Run router + inspect
Change `rerun=True` to re-route all complaints including already routed ones.

In [ ]:
import pandas as pd

decisions = run_router(rerun=False)

# ── query routing_decisions ───────────────────────────────────────────────────
rtr = duckdb.connect(ROUTING_DB, read_only=True)

print("\n── All routing decisions ─────────────────────────────────")
display(rtr.execute("""
    SELECT routing_id, complaint_id, cust_name, category, priority,
           complaint_created_at, original_team, routed_team,
           escalate, confidence, sla_hours, routed_at
    FROM routing_decisions
    ORDER BY priority ASC, complaint_created_at ASC
""").df())

print("\n── Escalations ───────────────────────────────────────────")
display(rtr.execute("""
    SELECT complaint_id, cust_name, priority, routed_team,
           escalate_to, action, complaint_created_at
    FROM routing_decisions
    WHERE escalate = true
    ORDER BY priority ASC
""").df())

print("\n── Team changed by agent ─────────────────────────────────")
display(rtr.execute("""
    SELECT complaint_id, complaint_created_at,
           original_team, routed_team, routing_reason
    FROM routing_decisions
    WHERE routed_team != original_team
""").df())

print("\n── Summary by team ───────────────────────────────────────")
display(rtr.execute("""
    SELECT routed_team,
           COUNT(*)                                            AS total,
           SUM(CASE WHEN escalate       THEN 1 ELSE 0 END)    AS escalations,
           SUM(CASE WHEN confidence='high'   THEN 1 ELSE 0 END) AS high_conf,
           SUM(CASE WHEN confidence='low'    THEN 1 ELSE 0 END) AS low_conf,
           ROUND(AVG(latency_ms), 0)                          AS avg_ms
    FROM routing_decisions
    GROUP BY routed_team
    ORDER BY total DESC
""").df())

rtr.close()